# 05. 멀티에이전트 실전 + Streamlit 프로젝트 템플릿

## 학습 목표
- Tool, Agent, Supervisor, StateGraph를 통합한 멀티에이전트 시스템을 구현할 수 있다
- 노트북에서 전체 시스템을 테스트할 수 있다
- Streamlit을 활용하여 멀티에이전트 시스템을 웹 앱으로 배포할 수 있다

## 1. 환경 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env 파일에 설정되어 있는지 확인하세요!"

print("환경 설정 완료!")
print("이 노트북에서는 멀티에이전트 시스템을 통합하고, Streamlit 앱으로 변환합니다.")

## 2. 전체 시스템 코드 (Tool + Agent + Supervisor + StateGraph 통합)

이전 노트북에서 배운 모든 것을 하나의 완성된 시스템으로 통합합니다.

### 시스템 구조
```
사용자 입력
    ↓
Supervisor (라우팅 결정)
    ├→ Search Agent (정보 검색)
    ├→ Analysis Agent (데이터 분석)
    └→ Writer Agent (보고서 작성)
    ↓
Supervisor (완료 판단)
    ↓
최종 응답
```

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START, END, add_messages

# ===== 1) LLM 설정 =====
llm = ChatOpenAI(model="gpt-5-mini")

# ===== 2) Tool 정의 =====
@tool
def search_web(query: str) -> str:
    """웹에서 최신 정보를 검색합니다."""
    # 실제 프로젝트에서는 Tavily, Serper 등의 검색 API를 사용하세요
    return f"[검색 결과] '{query}': AI 에이전트 기술이 빠르게 발전하며 기업용 솔루션이 확대되고 있습니다. 주요 프레임워크로 LangGraph, CrewAI, AutoGen 등이 있습니다."

@tool
def search_news(topic: str) -> str:
    """최신 뉴스를 검색합니다."""
    return f"[뉴스] '{topic}': 최근 기업들이 AI 에이전트를 업무 자동화에 적극 도입하고 있습니다."

@tool
def analyze_data(data: str) -> str:
    """데이터를 분석하고 인사이트를 도출합니다."""
    return f"[분석] 핵심 인사이트: 1) 에이전트 자동화 확산 2) 멀티에이전트 협업 패턴 부상 3) 안전성/신뢰성 강화 추세"

@tool
def compare_options(option_a: str, option_b: str) -> str:
    """두 옵션을 비교 분석합니다."""
    return f"[비교] '{option_a}' vs '{option_b}': 각각 고유한 강점이 있으며, 사용 목적에 따라 선택이 달라집니다."

@tool
def write_report(topic: str, content: str) -> str:
    """보고서를 작성합니다."""
    return f"[보고서] {topic}\n\n{content}\n\n결론: 해당 분야는 지속적인 성장이 예상됩니다."

@tool
def format_summary(text: str) -> str:
    """텍스트를 요약 형식으로 정리합니다."""
    return f"[요약] {text[:500]}"

# ===== 3) 워커 에이전트 생성 =====
search_agent = create_react_agent(
    llm, [search_web, search_news],
    prompt="당신은 정보 검색 전문가입니다. 주어진 주제에 대해 웹과 뉴스를 검색하여 정보를 수집하세요. 한국어로 답변하세요."
)

analysis_agent = create_react_agent(
    llm, [analyze_data, compare_options],
    prompt="당신은 데이터 분석 전문가입니다. 수집된 정보를 분석하고 인사이트를 도출하세요. 한국어로 답변하세요."
)

writer_agent = create_react_agent(
    llm, [write_report, format_summary],
    prompt="당신은 보고서 작성 전문가입니다. 분석 결과를 바탕으로 깔끔한 보고서를 작성하세요. 한국어로 답변하세요."
)

# ===== 4) Supervisor 설정 =====
class RouteDecision(BaseModel):
    next_agent: Literal["search", "analysis", "writer", "FINISH"] = Field(
        description="다음에 작업할 에이전트"
    )
    instruction: str = Field(
        description="에이전트에게 전달할 지시사항"
    )

supervisor_llm = llm.with_structured_output(RouteDecision)

SUPERVISOR_PROMPT = """당신은 멀티에이전트 팀의 Supervisor입니다.
사용자의 요청과 현재까지의 작업 결과를 보고, 다음 에이전트를 결정하세요.

팀원:
- search: 웹/뉴스 검색으로 정보 수집
- analysis: 수집된 정보 분석 및 인사이트 도출
- writer: 보고서 작성 및 요약
- FINISH: 모든 작업 완료

일반적 흐름: search -> analysis -> writer -> FINISH
단순 질문은 적절한 에이전트에게 바로 보내세요."""

# ===== 5) State 정의 =====
class MultiAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str
    agent_outputs: dict

# ===== 6) 노드 함수 정의 =====
def supervisor_node(state: MultiAgentState) -> dict:
    agent_outputs = state.get("agent_outputs", {})
    context = ""
    if agent_outputs:
        context = "\n\n현재까지의 작업 결과:\n"
        for name, output in agent_outputs.items():
            context += f"- {name}: {output[:200]}\n"
    
    messages = [
        SystemMessage(content=SUPERVISOR_PROMPT + context),
        *state["messages"]
    ]
    decision = supervisor_llm.invoke(messages)
    print(f"  [Supervisor] -> {decision.next_agent}: {decision.instruction}")
    
    return {
        "next_agent": decision.next_agent,
        "messages": [AIMessage(content=f"[Supervisor -> {decision.next_agent}] {decision.instruction}")]
    }

def make_worker_node(agent, agent_name):
    """워커 에이전트 노드 팩토리"""
    def worker_node(state: MultiAgentState) -> dict:
        result = agent.invoke({"messages": state["messages"]})
        output = result["messages"][-1].content
        print(f"  [{agent_name} Agent] 완료")
        
        agent_outputs = state.get("agent_outputs", {})
        agent_outputs[agent_name] = output
        
        return {
            "messages": [AIMessage(content=f"[{agent_name} 결과] {output}")],
            "agent_outputs": agent_outputs
        }
    return worker_node

def route_to_agent(state: MultiAgentState) -> str:
    next_agent = state["next_agent"]
    return END if next_agent == "FINISH" else next_agent

# ===== 7) 그래프 구성 =====
builder = StateGraph(MultiAgentState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("search", make_worker_node(search_agent, "Search"))
builder.add_node("analysis", make_worker_node(analysis_agent, "Analysis"))
builder.add_node("writer", make_worker_node(writer_agent, "Writer"))

builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor", route_to_agent,
    {"search": "search", "analysis": "analysis", "writer": "writer", END: END}
)
builder.add_edge("search", "supervisor")
builder.add_edge("analysis", "supervisor")
builder.add_edge("writer", "supervisor")

multi_agent_system = builder.compile()

print("멀티에이전트 시스템 구성 완료!")
multi_agent_system.get_graph().print_ascii()

## 3. 노트북에서 테스트

In [ ]:
# 테스트 함수
def run_multi_agent(query: str) -> str:
    """멀티에이전트 시스템을 실행하고 최종 결과를 반환합니다."""
    print(f"\n{'='*60}")
    print(f"질문: {query}")
    print(f"{'='*60}")
    
    result = multi_agent_system.invoke(
        {
            "messages": [HumanMessage(content=query)],
            "next_agent": "",
            "agent_outputs": {}
        },
        config={"recursion_limit": 20}
    )
    
    final_answer = result["messages"][-1].content
    print(f"\n--- 최종 응답 ---")
    print(final_answer)
    return final_answer

# 테스트 실행
run_multi_agent("AI 에이전트 기술 동향을 조사하고 분석 보고서를 작성해주세요.")

In [ ]:
# 추가 테스트
run_multi_agent("LangGraph와 CrewAI를 비교 분석해주세요.")

## 4. Streamlit app.py 코드 생성

아래 셀을 실행하면 `app.py` 파일이 생성됩니다.

In [ ]:
%%writefile app.py
import streamlit as st
from dotenv import load_dotenv
load_dotenv()

import os
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START, END, add_messages

# ===== 페이지 설정 =====
st.set_page_config(
    page_title="Multi-Agent System",
    page_icon="🤖",
    layout="wide"
)

st.title("멀티에이전트 시스템")
st.caption("Supervisor 패턴 기반 멀티에이전트 시스템 (Search + Analysis + Writer)")

# ===== 시스템 초기화 (캐싱) =====
@st.cache_resource
def build_multi_agent_system():
    """멀티에이전트 시스템을 구성합니다 (한 번만 실행)"""
    
    llm = ChatOpenAI(model="gpt-5-mini")
    
    # Tool 정의
    @tool
    def search_web(query: str) -> str:
        """웹에서 최신 정보를 검색합니다."""
        return f"[검색 결과] '{query}': AI 에이전트 기술이 빠르게 발전하며 기업용 솔루션이 확대되고 있습니다."

    @tool
    def search_news(topic: str) -> str:
        """최신 뉴스를 검색합니다."""
        return f"[뉴스] '{topic}': 최근 기업들이 AI 에이전트를 업무 자동화에 적극 도입하고 있습니다."

    @tool
    def analyze_data(data: str) -> str:
        """데이터를 분석합니다."""
        return f"[분석] 핵심 인사이트: 1) 에이전트 자동화 확산 2) 멀티에이전트 협업 패턴 부상 3) 안전성 강화 추세"

    @tool
    def compare_options(option_a: str, option_b: str) -> str:
        """두 옵션을 비교합니다."""
        return f"[비교] '{option_a}' vs '{option_b}': 각각 고유한 강점이 있습니다."

    @tool
    def write_report(topic: str, content: str) -> str:
        """보고서를 작성합니다."""
        return f"[보고서] {topic}\n\n{content}\n\n결론: 해당 분야는 지속적인 성장이 예상됩니다."

    @tool
    def format_summary(text: str) -> str:
        """텍스트를 요약합니다."""
        return f"[요약] {text[:500]}"

    # 워커 에이전트
    search_agent = create_react_agent(
        llm, [search_web, search_news],
        prompt="당신은 정보 검색 전문가입니다. 한국어로 답변하세요."
    )
    analysis_agent = create_react_agent(
        llm, [analyze_data, compare_options],
        prompt="당신은 데이터 분석 전문가입니다. 한국어로 답변하세요."
    )
    writer_agent = create_react_agent(
        llm, [write_report, format_summary],
        prompt="당신은 보고서 작성 전문가입니다. 한국어로 답변하세요."
    )

    # Supervisor
    class RouteDecision(BaseModel):
        next_agent: Literal["search", "analysis", "writer", "FINISH"] = Field(
            description="다음에 작업할 에이전트"
        )
        instruction: str = Field(description="에이전트에게 전달할 지시사항")

    supervisor_llm = llm.with_structured_output(RouteDecision)

    SUPERVISOR_PROMPT = """당신은 멀티에이전트 팀의 Supervisor입니다.
팀원: search(검색), analysis(분석), writer(보고서), FINISH(완료)
일반적 흐름: search -> analysis -> writer -> FINISH"""

    # State
    class MultiAgentState(TypedDict):
        messages: Annotated[list, add_messages]
        next_agent: str
        agent_outputs: dict
        log: list  # 실행 로그

    # 노드
    def supervisor_node(state: MultiAgentState) -> dict:
        agent_outputs = state.get("agent_outputs", {})
        context = ""
        if agent_outputs:
            context = "\n\n현재까지의 작업 결과:\n"
            for name, output in agent_outputs.items():
                context += f"- {name}: {output[:200]}\n"
        
        messages = [
            SystemMessage(content=SUPERVISOR_PROMPT + context),
            *state["messages"]
        ]
        decision = supervisor_llm.invoke(messages)
        
        log = state.get("log", [])
        log.append(f"Supervisor -> {decision.next_agent}: {decision.instruction}")
        
        return {
            "next_agent": decision.next_agent,
            "messages": [AIMessage(content=f"[Supervisor -> {decision.next_agent}] {decision.instruction}")],
            "log": log
        }

    def make_worker_node(agent, agent_name):
        def worker_node(state: MultiAgentState) -> dict:
            result = agent.invoke({"messages": state["messages"]})
            output = result["messages"][-1].content
            
            agent_outputs = state.get("agent_outputs", {})
            agent_outputs[agent_name] = output
            
            log = state.get("log", [])
            log.append(f"{agent_name} Agent 완료")
            
            return {
                "messages": [AIMessage(content=f"[{agent_name} 결과] {output}")],
                "agent_outputs": agent_outputs,
                "log": log
            }
        return worker_node

    def route_to_agent(state: MultiAgentState) -> str:
        return END if state["next_agent"] == "FINISH" else state["next_agent"]

    # 그래프 구성
    builder = StateGraph(MultiAgentState)
    builder.add_node("supervisor", supervisor_node)
    builder.add_node("search", make_worker_node(search_agent, "Search"))
    builder.add_node("analysis", make_worker_node(analysis_agent, "Analysis"))
    builder.add_node("writer", make_worker_node(writer_agent, "Writer"))

    builder.add_edge(START, "supervisor")
    builder.add_conditional_edges(
        "supervisor", route_to_agent,
        {"search": "search", "analysis": "analysis", "writer": "writer", END: END}
    )
    builder.add_edge("search", "supervisor")
    builder.add_edge("analysis", "supervisor")
    builder.add_edge("writer", "supervisor")

    return builder.compile()

# 시스템 빌드
multi_agent_system = build_multi_agent_system()

# ===== 세션 상태 초기화 =====
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# ===== 사이드바 =====
with st.sidebar:
    st.header("시스템 정보")
    st.info(
        "이 시스템은 Supervisor 패턴 기반의\n"
        "멀티에이전트 시스템입니다.\n\n"
        "- Search Agent: 정보 검색\n"
        "- Analysis Agent: 데이터 분석\n"
        "- Writer Agent: 보고서 작성"
    )
    
    if st.button("대화 초기화"):
        st.session_state.chat_history = []
        st.rerun()

# ===== 대화 히스토리 표시 =====
for msg in st.session_state.chat_history:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ===== 사용자 입력 =====
if user_input := st.chat_input("질문을 입력하세요..."):
    # 사용자 메시지 표시
    st.session_state.chat_history.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    
    # 에이전트 실행
    with st.chat_message("assistant"):
        with st.spinner("에이전트가 작업 중입니다..."):
            result = multi_agent_system.invoke(
                {
                    "messages": [HumanMessage(content=user_input)],
                    "next_agent": "",
                    "agent_outputs": {},
                    "log": []
                },
                config={"recursion_limit": 20}
            )
            
            # 실행 로그 표시
            log = result.get("log", [])
            if log:
                with st.expander("실행 로그 보기"):
                    for entry in log:
                        st.text(entry)
            
            # 최종 응답
            final_answer = result["messages"][-1].content
            st.markdown(final_answer)
    
    # 히스토리에 추가
    st.session_state.chat_history.append({"role": "assistant", "content": final_answer})

## 5. 실행 방법

### Streamlit 앱 실행

터미널에서 아래 명령어를 실행하세요:

```bash
# Streamlit 설치 (최초 1회)
pip install streamlit

# 앱 실행
streamlit run app.py
```

### 프로젝트 확장 아이디어

1. **실제 검색 API 연동**: Tavily, Serper 등의 검색 API로 `search_web` Tool 교체
2. **RAG 통합**: ChromaDB에 문서를 저장하고, 검색 에이전트에 retriever 연결
3. **새 에이전트 추가**: 번역 에이전트, 코드 생성 에이전트 등
4. **Human-in-the-Loop**: Supervisor 결정 전에 사용자 확인 단계 추가
5. **스트리밍 응답**: LangGraph의 stream 기능으로 실시간 응답 표시

In [ ]:
# Streamlit 앱을 노트북에서 바로 실행하려면 아래 주석을 해제하세요
# !streamlit run app.py